# Smart Radio Frequency Spectrum Forecasting and Anomaly Detection for Zimbabwe's Telecom Networks


## 1. Environment Setup

Install and import all required libraries. This notebook requires TensorFlow, scikit-learn, and standard data science packages.

In [ ]:
# Install required packages (uncomment if needed)
# !pip install tensorflow scikit-learn pandas numpy matplotlib seaborn

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import json
import os
import warnings
from datetime import datetime, timedelta

warnings.filterwarnings('ignore')
np.random.seed(42)

# Set plotting style
plt.style.use('seaborn-v0_8-darkgrid')
plt.rcParams['figure.figsize'] = (14, 6)
plt.rcParams['font.size'] = 12
plt.rcParams['axes.titlesize'] = 14
plt.rcParams['axes.labelsize'] = 12

# Create output directories
os.makedirs('data', exist_ok=True)
os.makedirs('models', exist_ok=True)
os.makedirs('figures', exist_ok=True)

print("Environment ready.")
print(f"NumPy: {np.__version__}")
print(f"Pandas: {pd.__version__}")

## 2. Synthetic Dataset Generation

We generate a realistic multivariate, geo-temporal time series dataset simulating radio frequency spectrum usage across Zimbabwe's major urban centres. The dataset models:

- **13 cell towers** across Harare, Bulawayo, Chitungwiza, and rural areas
- **5 frequency bands** (900 MHz to 3500 MHz) used by Zimbabwean MNOs
- **Hourly measurements** over 3 years (Jan 2021 – Dec 2023)
- **Realistic patterns**: diurnal cycles, weekly seasonality, growth trends, and density-based traffic
- **2% injected anomalies**: interference, traffic surges, signal dropouts, and unauthorized transmissions

In [ ]:
# ============================================================
# 2.1 Define Zimbabwe Cell Tower Locations
# ============================================================

cell_towers = [
    # Harare - High Density Urban
    {"tower_id": "HRE-CBD-001", "lat": -17.8292, "lon": 31.0522, "area": "Harare CBD", "density": "high_urban"},
    {"tower_id": "HRE-CBD-002", "lat": -17.8310, "lon": 31.0480, "area": "Harare CBD", "density": "high_urban"},
    {"tower_id": "HRE-AVD-003", "lat": -17.8015, "lon": 31.0440, "area": "Harare Avondale", "density": "high_urban"},
    {"tower_id": "HRE-MSA-004", "lat": -17.8560, "lon": 31.0300, "area": "Harare Mbare", "density": "high_urban"},
    {"tower_id": "HRE-BRW-005", "lat": -17.7880, "lon": 31.0820, "area": "Harare Borrowdale", "density": "high_urban"},
    # Bulawayo - High Density Urban
    {"tower_id": "BYO-CBD-001", "lat": -20.1500, "lon": 28.5800, "area": "Bulawayo CBD", "density": "high_urban"},
    {"tower_id": "BYO-HIL-002", "lat": -20.1350, "lon": 28.6100, "area": "Bulawayo Hillside", "density": "high_urban"},
    # Peri-Urban
    {"tower_id": "CHT-PU-001", "lat": -18.9700, "lon": 29.8200, "area": "Chitungwiza", "density": "peri_urban"},
    {"tower_id": "GWR-PU-001", "lat": -18.1800, "lon": 31.5900, "area": "Goromonzi", "density": "peri_urban"},
    {"tower_id": "NRT-PU-001", "lat": -17.3700, "lon": 30.6800, "area": "Norton", "density": "peri_urban"},
    # Rural
    {"tower_id": "MUT-RU-001", "lat": -18.9700, "lon": 32.6700, "area": "Mutare Rural", "density": "rural"},
    {"tower_id": "GKW-RU-001", "lat": -21.0100, "lon": 29.1500, "area": "Gwanda Rural", "density": "rural"},
    {"tower_id": "KRM-RU-001", "lat": -16.6300, "lon": 28.8000, "area": "Kariba Rural", "density": "rural"},
]

# Frequency bands used in Zimbabwe's telecom networks
frequency_bands = [
    {"band": "900 MHz",  "technology": "2G/3G",  "base_occupancy": 0.65},
    {"band": "1800 MHz", "technology": "2G/4G",  "base_occupancy": 0.55},
    {"band": "2100 MHz", "technology": "3G/4G",  "base_occupancy": 0.50},
    {"band": "2600 MHz", "technology": "4G LTE", "base_occupancy": 0.40},
    {"band": "3500 MHz", "technology": "5G NR",  "base_occupancy": 0.20},
]

# Mobile Network Operators
operators = ["Econet", "NetOne", "Telecel"]

print(f"Cell Towers: {len(cell_towers)}")
print(f"Frequency Bands: {len(frequency_bands)}")
print(f"Operators: {operators}")
print(f"Total combinations: {len(cell_towers) * len(frequency_bands)}")

In [ ]:
# ============================================================
# 2.2 Helper Functions for Realistic Pattern Generation
# ============================================================

def get_density_multiplier(density):
    """Traffic multiplier based on area density type."""
    return {"high_urban": 1.0, "peri_urban": 0.55, "rural": 0.25}[density]

def get_hourly_pattern(hour):
    """Realistic diurnal traffic pattern for Zimbabwe.
    Low at night, morning peak, lunch dip, afternoon peak, evening decline."""
    pattern = {
        0: 0.15, 1: 0.10, 2: 0.08, 3: 0.07, 4: 0.08, 5: 0.12,
        6: 0.25, 7: 0.45, 8: 0.70, 9: 0.85, 10: 0.90, 11: 0.88,
        12: 0.80, 13: 0.82, 14: 0.85, 15: 0.88, 16: 0.92, 17: 0.95,
        18: 1.00, 19: 0.95, 20: 0.85, 21: 0.70, 22: 0.50, 23: 0.30
    }
    return pattern[hour]

def get_day_of_week_factor(dow):
    """Day of week traffic factor (0=Monday)."""
    factors = {0: 0.95, 1: 0.97, 2: 1.0, 3: 0.98, 4: 1.05, 5: 1.10, 6: 0.90}
    return factors[dow]

def inject_anomalies(df, anomaly_rate=0.02):
    """Inject realistic anomalies into the dataset."""
    n = len(df)
    n_anomalies = int(n * anomaly_rate)
    anomaly_indices = np.random.choice(n, n_anomalies, replace=False)
    
    df["is_anomaly"] = 0
    df["anomaly_type"] = "normal"
    
    for idx in anomaly_indices:
        atype = np.random.choice(["interference", "surge", "dropout", "unauthorized"])
        
        if atype == "interference":
            df.loc[idx, "rssi_dbm"] = np.clip(df.loc[idx, "rssi_dbm"] + np.random.uniform(15, 35), -120, -20)
            df.loc[idx, "channel_occupancy"] = min(df.loc[idx, "channel_occupancy"] * 1.5, 1.0)
            df.loc[idx, "anomaly_type"] = "interference"
        elif atype == "surge":
            df.loc[idx, "traffic_volume_gb"] *= np.random.uniform(2.5, 5.0)
            df.loc[idx, "active_users"] = int(df.loc[idx, "active_users"] * np.random.uniform(2.0, 4.0))
            df.loc[idx, "channel_occupancy"] = min(df.loc[idx, "channel_occupancy"] * 2.0, 1.0)
            df.loc[idx, "anomaly_type"] = "traffic_surge"
        elif atype == "dropout":
            df.loc[idx, "rssi_dbm"] = np.random.uniform(-115, -105)
            df.loc[idx, "traffic_volume_gb"] *= 0.1
            df.loc[idx, "active_users"] = int(df.loc[idx, "active_users"] * 0.1)
            df.loc[idx, "channel_occupancy"] *= 0.15
            df.loc[idx, "anomaly_type"] = "signal_dropout"
        else:
            df.loc[idx, "rssi_dbm"] = np.clip(df.loc[idx, "rssi_dbm"] + np.random.uniform(10, 25), -120, -20)
            df.loc[idx, "channel_occupancy"] = min(df.loc[idx, "channel_occupancy"] + 0.3, 1.0)
            df.loc[idx, "anomaly_type"] = "unauthorized_tx"
        
        df.loc[idx, "is_anomaly"] = 1
    
    return df

# Visualize the hourly pattern
hours = list(range(24))
pattern_vals = [get_hourly_pattern(h) for h in hours]

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].plot(hours, pattern_vals, 'o-', color='#00d4aa', linewidth=2, markersize=6)
axes[0].fill_between(hours, pattern_vals, alpha=0.2, color='#00d4aa')
axes[0].set_xlabel('Hour of Day')
axes[0].set_ylabel('Traffic Multiplier')
axes[0].set_title('Diurnal Traffic Pattern (Zimbabwe)')
axes[0].set_xticks(range(0, 24, 3))

dow_labels = ['Mon', 'Tue', 'Wed', 'Thu', 'Fri', 'Sat', 'Sun']
dow_vals = [get_day_of_week_factor(d) for d in range(7)]
axes[1].bar(dow_labels, dow_vals, color=['#0891b2']*5 + ['#f59e0b', '#ef4444'], edgecolor='white')
axes[1].set_ylabel('Traffic Factor')
axes[1].set_title('Day-of-Week Traffic Pattern')

plt.tight_layout()
plt.savefig('figures/traffic_patterns.png', dpi=150, bbox_inches='tight')
plt.show()
print("Figure saved: figures/traffic_patterns.png")

In [ ]:
# ============================================================
# 2.3 Generate the Full Dataset
# ============================================================

print("=" * 60)
print("GENERATING ZIMBABWE TELECOM SPECTRUM DATASET")
print("=" * 60)

# Time range: 36 months (Jan 2021 - Dec 2023), hourly intervals
start_date = datetime(2021, 1, 1)
end_date = datetime(2023, 12, 31, 23, 0, 0)
timestamps = pd.date_range(start=start_date, end=end_date, freq="H")
print(f"Time range: {start_date} to {end_date}")
print(f"Total timestamps: {len(timestamps)}")

records = []
total_combos = len(cell_towers) * len(frequency_bands)
combo_count = 0

for tower in cell_towers:
    for band in frequency_bands:
        combo_count += 1
        operator = np.random.choice(operators)
        density_mult = get_density_multiplier(tower["density"])
        
        if combo_count % 10 == 0:
            print(f"  Generating combo {combo_count}/{total_combos}: {tower['tower_id']} - {band['band']}")
        
        for ts in timestamps:
            hour = ts.hour
            dow = ts.dayofweek
            month = ts.month
            
            # Hourly + daily + seasonal patterns
            hourly = get_hourly_pattern(hour)
            daily = get_day_of_week_factor(dow)
            seasonal = 1.0 + 0.08 * np.sin(2 * np.pi * (month - 1) / 12)
            days_elapsed = (ts - start_date).days
            growth = 1.0 + 0.0002 * days_elapsed
            
            combined = hourly * daily * seasonal * density_mult * growth
            
            # RSSI (dBm)
            base_rssi = -55 if tower["density"] == "high_urban" else (-70 if tower["density"] == "peri_urban" else -85)
            rssi = base_rssi - 10 * (1 - combined) + np.random.normal(0, 3)
            rssi = np.clip(rssi, -110, -30)
            
            # Channel Occupancy (0-1)
            occupancy = band["base_occupancy"] * combined + np.random.normal(0, 0.05)
            occupancy = np.clip(occupancy, 0.01, 0.99)
            
            # Traffic Volume (GB)
            base_traffic = 50 if tower["density"] == "high_urban" else (20 if tower["density"] == "peri_urban" else 5)
            traffic = base_traffic * combined + np.random.exponential(2)
            traffic = max(0.1, traffic)
            
            # Active Users
            base_users = 500 if tower["density"] == "high_urban" else (150 if tower["density"] == "peri_urban" else 30)
            users = int(base_users * combined + np.random.poisson(10))
            users = max(1, users)
            
            records.append({
                "timestamp": ts,
                "tower_id": tower["tower_id"],
                "area": tower["area"],
                "latitude": tower["lat"],
                "longitude": tower["lon"],
                "density_type": tower["density"],
                "frequency_band": band["band"],
                "technology": band["technology"],
                "operator": operator,
                "rssi_dbm": round(rssi, 2),
                "channel_occupancy": round(occupancy, 4),
                "traffic_volume_gb": round(traffic, 3),
                "active_users": users,
            })

print(f"\nTotal raw records generated: {len(records):,}")

# Create DataFrame
df = pd.DataFrame(records)

# Inject anomalies
print("\nInjecting anomalies (2% rate)...")
df = inject_anomalies(df, anomaly_rate=0.02)
anomaly_count = df["is_anomaly"].sum()
print(f"Anomalies injected: {anomaly_count:,} ({anomaly_count/len(df)*100:.2f}%)")

# Save full dataset
df.to_csv("data/zimbabwe_spectrum_full.csv", index=False)
print(f"\nFull dataset saved: data/zimbabwe_spectrum_full.csv")
print(f"Shape: {df.shape}")

# Create sample subset for model training
sample = df[df["tower_id"].isin(["HRE-CBD-001", "BYO-CBD-001", "CHT-PU-001", "MUT-RU-001"])]
sample = sample[sample["frequency_band"].isin(["900 MHz", "1800 MHz", "2600 MHz"])]
sample.to_csv("data/zimbabwe_spectrum_sample.csv", index=False)
print(f"Sample dataset saved: data/zimbabwe_spectrum_sample.csv")
print(f"Sample shape: {sample.shape}")

print("\n" + "=" * 60)
print("DATASET GENERATION COMPLETE")
print("=" * 60)

## 3. Exploratory Data Analysis (EDA)

Before training models, we conduct a thorough exploration of the dataset to understand distributions, temporal patterns, and the characteristics of injected anomalies.

In [ ]:
# ============================================================
# 3.1 Dataset Overview
# ============================================================

print("Dataset Shape:", df.shape)
print("\nColumn Types:")
print(df.dtypes)
print("\nBasic Statistics:")
df[["rssi_dbm", "channel_occupancy", "traffic_volume_gb", "active_users"]].describe().round(3)

In [ ]:
# ============================================================
# 3.2 Distribution of Key Features
# ============================================================

fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# RSSI Distribution
axes[0, 0].hist(df["rssi_dbm"], bins=80, color='#0891b2', alpha=0.8, edgecolor='white')
axes[0, 0].set_title("RSSI Distribution (dBm)")
axes[0, 0].set_xlabel("RSSI (dBm)")
axes[0, 0].set_ylabel("Frequency")

# Channel Occupancy
axes[0, 1].hist(df["channel_occupancy"], bins=80, color='#f59e0b', alpha=0.8, edgecolor='white')
axes[0, 1].set_title("Channel Occupancy Distribution")
axes[0, 1].set_xlabel("Occupancy (0-1)")
axes[0, 1].set_ylabel("Frequency")

# Traffic Volume
axes[1, 0].hist(df["traffic_volume_gb"], bins=80, color='#10b981', alpha=0.8, edgecolor='white')
axes[1, 0].set_title("Traffic Volume Distribution (GB)")
axes[1, 0].set_xlabel("Traffic (GB)")
axes[1, 0].set_ylabel("Frequency")

# Active Users
axes[1, 1].hist(df["active_users"], bins=80, color='#ef4444', alpha=0.8, edgecolor='white')
axes[1, 1].set_title("Active Users Distribution")
axes[1, 1].set_xlabel("Active Users")
axes[1, 1].set_ylabel("Frequency")

plt.suptitle("Feature Distributions Across All Towers and Bands", fontsize=16, y=1.02)
plt.tight_layout()
plt.savefig('figures/feature_distributions.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# ============================================================
# 3.3 Traffic Patterns by Tower Density Type
# ============================================================

fig, axes = plt.subplots(1, 3, figsize=(16, 5))

for i, density in enumerate(["high_urban", "peri_urban", "rural"]):
    subset = df[df["density_type"] == density]
    hourly = subset.groupby(pd.to_datetime(subset["timestamp"]).dt.hour)["traffic_volume_gb"].mean()
    axes[i].plot(hourly.index, hourly.values, 'o-', linewidth=2, markersize=4, color=['#0891b2', '#f59e0b', '#10b981'][i])
    axes[i].fill_between(hourly.index, hourly.values, alpha=0.15, color=['#0891b2', '#f59e0b', '#10b981'][i])
    axes[i].set_title(f"{density.replace('_', ' ').title()}")
    axes[i].set_xlabel("Hour of Day")
    axes[i].set_ylabel("Avg Traffic (GB)")
    axes[i].set_xticks(range(0, 24, 4))

plt.suptitle("Average Hourly Traffic by Density Type", fontsize=14, y=1.02)
plt.tight_layout()
plt.savefig('figures/traffic_by_density.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# ============================================================
# 3.4 Anomaly Analysis
# ============================================================

fig, axes = plt.subplots(1, 3, figsize=(16, 5))

# Anomaly type distribution
anomaly_types = df[df["is_anomaly"] == 1]["anomaly_type"].value_counts()
colors_anom = ['#ef4444', '#f59e0b', '#8b5cf6', '#0891b2']
axes[0].barh(anomaly_types.index, anomaly_types.values, color=colors_anom, edgecolor='white')
axes[0].set_title("Anomaly Types Distribution")
axes[0].set_xlabel("Count")

# Normal vs Anomaly traffic comparison
normal_traffic = df[df["is_anomaly"] == 0]["traffic_volume_gb"]
anomaly_traffic = df[df["is_anomaly"] == 1]["traffic_volume_gb"]
axes[1].boxplot([normal_traffic, anomaly_traffic], labels=["Normal", "Anomaly"],
                patch_artist=True, boxprops=dict(facecolor='#0891b2', alpha=0.7))
axes[1].set_title("Traffic Volume: Normal vs Anomaly")
axes[1].set_ylabel("Traffic (GB)")

# Anomaly RSSI comparison
normal_rssi = df[df["is_anomaly"] == 0]["rssi_dbm"]
anomaly_rssi = df[df["is_anomaly"] == 1]["rssi_dbm"]
axes[2].boxplot([normal_rssi, anomaly_rssi], labels=["Normal", "Anomaly"],
                patch_artist=True, boxprops=dict(facecolor='#f59e0b', alpha=0.7))
axes[2].set_title("RSSI: Normal vs Anomaly")
axes[2].set_ylabel("RSSI (dBm)")

plt.suptitle("Anomaly Characteristics Analysis", fontsize=14, y=1.02)
plt.tight_layout()
plt.savefig('figures/anomaly_analysis.png', dpi=150, bbox_inches='tight')
plt.show()

print(f"\nAnomaly Distribution:")
print(anomaly_types)
print(f"\nTotal anomalies: {df['is_anomaly'].sum():,} out of {len(df):,} records ({df['is_anomaly'].mean()*100:.2f}%)")

In [ ]:
# ============================================================
# 3.5 Correlation Heatmap
# ============================================================

numeric_cols = ["rssi_dbm", "channel_occupancy", "traffic_volume_gb", "active_users"]
corr = df[numeric_cols].corr()

fig, ax = plt.subplots(figsize=(8, 6))
sns.heatmap(corr, annot=True, fmt=".3f", cmap="coolwarm", center=0,
            square=True, linewidths=1, ax=ax, vmin=-1, vmax=1)
ax.set_title("Feature Correlation Matrix", fontsize=14)
plt.tight_layout()
plt.savefig('figures/correlation_heatmap.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# ============================================================
# 3.6 Band Occupancy Comparison
# ============================================================

band_occ = df.groupby("frequency_band")["channel_occupancy"].agg(["mean", "std"]).sort_values("mean", ascending=True)

fig, ax = plt.subplots(figsize=(10, 5))
bars = ax.barh(band_occ.index, band_occ["mean"], xerr=band_occ["std"],
               color=['#0891b2', '#10b981', '#f59e0b', '#ef4444', '#8b5cf6'],
               edgecolor='white', capsize=5)
ax.set_xlabel("Mean Channel Occupancy")
ax.set_title("Average Channel Occupancy by Frequency Band")
ax.set_xlim(0, 1)

for bar, val in zip(bars, band_occ["mean"]):
    ax.text(val + 0.02, bar.get_y() + bar.get_height()/2, f'{val:.3f}',
            va='center', fontweight='bold')

plt.tight_layout()
plt.savefig('figures/band_occupancy.png', dpi=150, bbox_inches='tight')
plt.show()

## 4. CNN-LSTM Hybrid Model for Spectrum Traffic Forecasting

The CNN-LSTM architecture combines:
- **Convolutional layers** for extracting local temporal features and patterns
- **LSTM layers** for capturing long-range temporal dependencies
- **Dense layers** for multi-step ahead prediction

**Configuration:**
- Lookback window: 72 hours (3 days)
- Forecast horizon: 6 hours ahead
- Train/Val/Test split: 70/15/15 (chronological)

In [ ]:
# ============================================================
# 4.1 Data Preparation for Forecasting
# ============================================================

from sklearn.preprocessing import StandardScaler

# Load sample data
sample_df = pd.read_csv("data/zimbabwe_spectrum_sample.csv")
sample_df["timestamp"] = pd.to_datetime(sample_df["timestamp"])

# Focus on one tower+band for forecasting
train_df = sample_df[(sample_df["tower_id"] == "HRE-CBD-001") & 
                     (sample_df["frequency_band"] == "900 MHz")].copy()
train_df = train_df.sort_values("timestamp").reset_index(drop=True)
print(f"Training subset (HRE-CBD-001, 900 MHz): {train_df.shape}")

# Features
features = ["rssi_dbm", "channel_occupancy", "traffic_volume_gb", "active_users"]
target = "traffic_volume_gb"

# Add cyclical temporal features
train_df["hour_sin"] = np.sin(2 * np.pi * train_df["timestamp"].dt.hour / 24)
train_df["hour_cos"] = np.cos(2 * np.pi * train_df["timestamp"].dt.hour / 24)
train_df["dow_sin"] = np.sin(2 * np.pi * train_df["timestamp"].dt.dayofweek / 7)
train_df["dow_cos"] = np.cos(2 * np.pi * train_df["timestamp"].dt.dayofweek / 7)

feature_cols = features + ["hour_sin", "hour_cos", "dow_sin", "dow_cos"]
n_features = len(feature_cols)

# Standardize features
scaler = StandardScaler()
scaled_data = scaler.fit_transform(train_df[feature_cols].values)

print(f"Features: {feature_cols}")
print(f"Number of features: {n_features}")
print(f"Scaled data shape: {scaled_data.shape}")

In [ ]:
# ============================================================
# 4.2 Create Sequences for Time Series
# ============================================================

LOOKBACK = 72     # 3 days of hourly data
FORECAST_HORIZON = 6  # Predict next 6 hours

def create_sequences(data, lookback, horizon, target_idx=2):
    """Create sliding window sequences for time series forecasting."""
    X, y = [], []
    for i in range(lookback, len(data) - horizon):
        X.append(data[i - lookback:i])
        y.append(data[i:i + horizon, target_idx])
    return np.array(X), np.array(y)

X, y = create_sequences(scaled_data, LOOKBACK, FORECAST_HORIZON)
print(f"Total sequences: X={X.shape}, y={y.shape}")

# Chronological split: 70% train, 15% val, 15% test
n = len(X)
train_end = int(n * 0.70)
val_end = int(n * 0.85)

X_train, y_train = X[:train_end], y[:train_end]
X_val, y_val = X[train_end:val_end], y[train_end:val_end]
X_test, y_test = X[val_end:], y[val_end:]

print(f"\nChronological Split:")
print(f"  Train: {X_train.shape[0]:,} sequences ({X_train.shape[0]/n*100:.1f}%)")
print(f"  Val:   {X_val.shape[0]:,} sequences ({X_val.shape[0]/n*100:.1f}%)")
print(f"  Test:  {X_test.shape[0]:,} sequences ({X_test.shape[0]/n*100:.1f}%)")

In [ ]:
# ============================================================
# 4.3 Build CNN-LSTM Architecture
# ============================================================

import tensorflow as tf
from tensorflow.keras.models import Model
from tensorflow.keras.layers import (Input, Conv1D, MaxPooling1D, LSTM, Dense,
                                      Dropout, BatchNormalization)
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau

tf.random.set_seed(42)

# Model Architecture
inputs = Input(shape=(LOOKBACK, n_features), name='input_sequence')

# CNN Block: Local feature extraction
x = Conv1D(filters=64, kernel_size=3, activation='relu', padding='same', name='conv1d_1')(inputs)
x = BatchNormalization(name='bn_1')(x)
x = Conv1D(filters=32, kernel_size=3, activation='relu', padding='same', name='conv1d_2')(x)
x = BatchNormalization(name='bn_2')(x)
x = MaxPooling1D(pool_size=2, name='maxpool')(x)
x = Dropout(0.2, name='dropout_cnn')(x)

# LSTM Block: Temporal dependency modelling
x = LSTM(64, return_sequences=True, name='lstm_1')(x)
x = Dropout(0.2, name='dropout_lstm1')(x)
x = LSTM(32, return_sequences=False, name='lstm_2')(x)
x = Dropout(0.2, name='dropout_lstm2')(x)

# Output Block
x = Dense(32, activation='relu', name='dense_1')(x)
outputs = Dense(FORECAST_HORIZON, name='forecast_output')(x)

cnn_lstm_model = Model(inputs, outputs, name="CNN_LSTM_Forecaster")
cnn_lstm_model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=0.001),
    loss='mse',
    metrics=['mae']
)

cnn_lstm_model.summary()

In [ ]:
# ============================================================
# 4.4 Train the CNN-LSTM Model
# ============================================================

callbacks = [
    EarlyStopping(monitor='val_loss', patience=10, restore_best_weights=True, verbose=1),
    ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=5, min_lr=1e-6, verbose=1)
]

print("Training CNN-LSTM Forecasting Model...")
print("=" * 50)

history = cnn_lstm_model.fit(
    X_train, y_train,
    validation_data=(X_val, y_val),
    epochs=50,
    batch_size=64,
    callbacks=callbacks,
    verbose=1
)

print("\nTraining complete!")

In [ ]:
# ============================================================
# 4.5 Evaluate Forecasting Model
# ============================================================

# Test set evaluation
test_loss, test_mae = cnn_lstm_model.evaluate(X_test, y_test, verbose=0)
y_pred = cnn_lstm_model.predict(X_test, verbose=0)

# Inverse transform to real-world scale
target_idx = feature_cols.index(target)
y_test_real = y_test * scaler.scale_[target_idx] + scaler.mean_[target_idx]
y_pred_real = y_pred * scaler.scale_[target_idx] + scaler.mean_[target_idx]

# Calculate metrics
mae_real = np.mean(np.abs(y_test_real - y_pred_real))
rmse = np.sqrt(np.mean((y_test_real - y_pred_real) ** 2))
mape = np.mean(np.abs((y_test_real - y_pred_real) / (y_test_real + 1e-8))) * 100

print("=" * 50)
print("CNN-LSTM FORECASTING RESULTS (Real Scale)")
print("=" * 50)
print(f"  MAE:  {mae_real:.4f} GB")
print(f"  RMSE: {rmse:.4f} GB")
print(f"  MAPE: {mape:.2f}%")
print(f"  MSE (scaled): {test_loss:.6f}")

# Save model
cnn_lstm_model.save("models/cnn_lstm_forecaster.keras")
print("\nModel saved: models/cnn_lstm_forecaster.keras")

In [ ]:
# ============================================================
# 4.6 Visualize Forecasting Results
# ============================================================

fig, axes = plt.subplots(2, 2, figsize=(16, 10))

# Training Loss Curves
axes[0, 0].plot(history.history['loss'], label='Train Loss', color='#0891b2', linewidth=2)
axes[0, 0].plot(history.history['val_loss'], label='Val Loss', color='#f59e0b', linewidth=2, linestyle='--')
axes[0, 0].set_title('CNN-LSTM Training Loss (MSE)')
axes[0, 0].set_xlabel('Epoch')
axes[0, 0].set_ylabel('Loss')
axes[0, 0].legend()
axes[0, 0].grid(True, alpha=0.3)

# MAE Curves
axes[0, 1].plot(history.history['mae'], label='Train MAE', color='#0891b2', linewidth=2)
axes[0, 1].plot(history.history['val_mae'], label='Val MAE', color='#f59e0b', linewidth=2, linestyle='--')
axes[0, 1].set_title('CNN-LSTM Training MAE')
axes[0, 1].set_xlabel('Epoch')
axes[0, 1].set_ylabel('MAE')
axes[0, 1].legend()
axes[0, 1].grid(True, alpha=0.3)

# Actual vs Predicted (last 120 points)
n_show = 120
axes[1, 0].plot(y_test_real[:n_show, 0], label='Actual', color='#0891b2', linewidth=1.5, alpha=0.8)
axes[1, 0].plot(y_pred_real[:n_show, 0], label='Predicted', color='#ef4444', linewidth=1.5, linestyle='--', alpha=0.8)
axes[1, 0].set_title(f'Actual vs Predicted Traffic (First {n_show} Test Points)')
axes[1, 0].set_xlabel('Time Step')
axes[1, 0].set_ylabel('Traffic Volume (GB)')
axes[1, 0].legend()
axes[1, 0].grid(True, alpha=0.3)

# Error Distribution
errors = np.abs(y_test_real[:, 0] - y_pred_real[:, 0])
axes[1, 1].hist(errors, bins=50, color='#f59e0b', alpha=0.8, edgecolor='white')
axes[1, 1].axvline(x=mae_real, color='#ef4444', linestyle='--', linewidth=2, label=f'MAE = {mae_real:.2f} GB')
axes[1, 1].set_title('Absolute Error Distribution')
axes[1, 1].set_xlabel('Absolute Error (GB)')
axes[1, 1].set_ylabel('Frequency')
axes[1, 1].legend()

plt.suptitle('CNN-LSTM Forecasting Model Performance', fontsize=16, y=1.02)
plt.tight_layout()
plt.savefig('figures/cnn_lstm_results.png', dpi=150, bbox_inches='tight')
plt.show()

## 5. Variational Autoencoder (VAE) for Anomaly Detection

The VAE learns the distribution of **normal** spectrum behaviour. At inference time, data points with high reconstruction error are flagged as anomalies.

**Architecture:**
- Encoder: Input → 32 → 16 → Latent (μ, σ)
- Latent dimension: 4
- Decoder: Latent → 16 → 32 → Output
- Threshold: 99th percentile of validation reconstruction errors

In [ ]:
# ============================================================
# 5.1 Prepare Data for VAE
# ============================================================

from tensorflow.keras.layers import Lambda

# Use all towers and bands from the sample
vae_features = ["rssi_dbm", "channel_occupancy", "traffic_volume_gb", "active_users"]

# Separate normal and anomaly data
normal_df = sample_df[sample_df["is_anomaly"] == 0].copy()
anomaly_df = sample_df[sample_df["is_anomaly"] == 1].copy()
print(f"Normal records: {len(normal_df):,}")
print(f"Anomaly records: {len(anomaly_df):,}")

# Add temporal features
for d in [normal_df, anomaly_df]:
    d["hour_sin"] = np.sin(2 * np.pi * pd.to_datetime(d["timestamp"]).dt.hour / 24)
    d["hour_cos"] = np.cos(2 * np.pi * pd.to_datetime(d["timestamp"]).dt.hour / 24)

vae_feature_cols = vae_features + ["hour_sin", "hour_cos"]

# Scale using only normal data
vae_scaler = StandardScaler()
normal_scaled = vae_scaler.fit_transform(normal_df[vae_feature_cols].values)

# Split normal data chronologically
n_normal = len(normal_scaled)
train_end_vae = int(n_normal * 0.70)
val_end_vae = int(n_normal * 0.85)

X_train_vae = normal_scaled[:train_end_vae]
X_val_vae = normal_scaled[train_end_vae:val_end_vae]

# Scale anomaly data using the same scaler
anomaly_scaled = vae_scaler.transform(anomaly_df[vae_feature_cols].values)

print(f"\nVAE Data Splits:")
print(f"  Train (normal): {X_train_vae.shape}")
print(f"  Val (normal):   {X_val_vae.shape}")
print(f"  Anomaly test:   {anomaly_scaled.shape}")

In [ ]:
# ============================================================
# 5.2 Build VAE Architecture
# ============================================================

import keras.ops as ops

input_dim = len(vae_feature_cols)
latent_dim = 4

# --- Encoder ---
encoder_inputs = Input(shape=(input_dim,), name='encoder_input')
h = Dense(32, activation='relu', name='enc_dense1')(encoder_inputs)
h = Dense(16, activation='relu', name='enc_dense2')(h)
z_mean = Dense(latent_dim, name='z_mean')(h)
z_log_var = Dense(latent_dim, name='z_log_var')(h)

def sampling(args):
    """Reparameterization trick for VAE."""
    z_mean, z_log_var = args
    batch = tf.shape(z_mean)[0]
    dim = tf.shape(z_mean)[1]
    epsilon = tf.random.normal(shape=(batch, dim))
    return z_mean + tf.exp(0.5 * z_log_var) * epsilon

z = Lambda(sampling, output_shape=(latent_dim,), name='z')([z_mean, z_log_var])

encoder = Model(encoder_inputs, [z_mean, z_log_var, z], name='encoder')
print("ENCODER:")
encoder.summary()

# --- Decoder ---
decoder_inputs = Input(shape=(latent_dim,), name='decoder_input')
d = Dense(16, activation='relu', name='dec_dense1')(decoder_inputs)
d = Dense(32, activation='relu', name='dec_dense2')(d)
decoder_outputs = Dense(input_dim, activation='linear', name='dec_output')(d)

decoder = Model(decoder_inputs, decoder_outputs, name='decoder')
print("\nDECODER:")
decoder.summary()

In [ ]:
# ============================================================
# 5.3 Compile VAE with Custom Loss
# ============================================================

class VAELossLayer(tf.keras.layers.Layer):
    """Custom layer that computes and adds VAE loss (reconstruction + KL divergence)."""
    def call(self, inputs):
        x, x_decoded, z_mean_val, z_log_var_val = inputs
        # Reconstruction loss (MSE)
        reconstruction_loss = ops.mean(ops.square(x - x_decoded), axis=-1) * input_dim
        # KL divergence loss
        kl_loss = -0.5 * ops.sum(1 + z_log_var_val - ops.square(z_mean_val) - ops.exp(z_log_var_val), axis=-1)
        total_loss = ops.mean(reconstruction_loss + kl_loss)
        self.add_loss(total_loss)
        return x_decoded

# Build full VAE
z_mean_out, z_log_var_out, z_out = encoder(encoder_inputs)
vae_decoded = decoder(z_out)
loss_layer = VAELossLayer()
vae_output_final = loss_layer([encoder_inputs, vae_decoded, z_mean_out, z_log_var_out])
vae_model = Model(encoder_inputs, vae_output_final, name='VAE')
vae_model.compile(optimizer=tf.keras.optimizers.Adam(learning_rate=0.001))

print("VAE Model compiled successfully.")
print(f"Input dimension: {input_dim}")
print(f"Latent dimension: {latent_dim}")

In [ ]:
# ============================================================
# 5.4 Train the VAE
# ============================================================

vae_callbacks = [
    EarlyStopping(monitor='val_loss', patience=10, restore_best_weights=True, verbose=1),
]

print("Training VAE Anomaly Detection Model...")
print("=" * 50)

vae_history = vae_model.fit(
    X_train_vae, X_train_vae,
    validation_data=(X_val_vae, X_val_vae),
    epochs=50,
    batch_size=256,
    callbacks=vae_callbacks,
    verbose=1
)

print("\nVAE Training complete!")

In [ ]:
# ============================================================
# 5.5 Determine Anomaly Threshold and Evaluate
# ============================================================

# Calculate reconstruction errors on validation set
val_reconstructed = vae_model.predict(X_val_vae, verbose=0)
val_errors = np.mean(np.square(X_val_vae - val_reconstructed), axis=1)

# Set threshold at 99th percentile of normal validation errors
threshold = float(np.percentile(val_errors, 99))
print(f"Anomaly Threshold (99th percentile): {threshold:.6f}")
print(f"Val error stats: mean={np.mean(val_errors):.4f}, std={np.std(val_errors):.4f}, max={np.max(val_errors):.4f}")

# Evaluate on test data
X_test_normal = normal_scaled[val_end_vae:]
normal_reconstructed = vae_model.predict(X_test_normal, verbose=0)
normal_errors = np.mean(np.square(X_test_normal - normal_reconstructed), axis=1)

anomaly_reconstructed = vae_model.predict(anomaly_scaled, verbose=0)
anomaly_errors = np.mean(np.square(anomaly_scaled - anomaly_reconstructed), axis=1)

# Classification
normal_pred = (normal_errors > threshold).astype(int)
anomaly_pred = (anomaly_errors > threshold).astype(int)

# Confusion Matrix Metrics
TP = int(np.sum(anomaly_pred == 1))
FP = int(np.sum(normal_pred == 1))
TN = int(np.sum(normal_pred == 0))
FN = int(np.sum(anomaly_pred == 0))

precision = TP / (TP + FP) if (TP + FP) > 0 else 0
recall = TP / (TP + FN) if (TP + FN) > 0 else 0
f1 = 2 * precision * recall / (precision + recall) if (precision + recall) > 0 else 0
accuracy = (TP + TN) / (TP + TN + FP + FN)

print("\n" + "=" * 50)
print("VAE ANOMALY DETECTION RESULTS")
print("=" * 50)
print(f"  True Positives:  {TP:,}")
print(f"  False Positives: {FP:,}")
print(f"  True Negatives:  {TN:,}")
print(f"  False Negatives: {FN:,}")
print(f"  ─────────────────────────")
print(f"  Precision: {precision:.4f} ({precision*100:.1f}%)")
print(f"  Recall:    {recall:.4f} ({recall*100:.1f}%)")
print(f"  F1-Score:  {f1:.4f} ({f1*100:.1f}%)")
print(f"  Accuracy:  {accuracy:.4f} ({accuracy*100:.1f}%)")

# Save models
encoder.save("models/vae_encoder.keras")
decoder.save("models/vae_decoder.keras")
vae_model.save("models/vae_full.keras")
print("\nVAE models saved.")

In [ ]:
# ============================================================
# 5.6 Visualize VAE Results
# ============================================================

fig, axes = plt.subplots(2, 2, figsize=(16, 10))

# VAE Training Loss
axes[0, 0].plot(vae_history.history['loss'], label='Train Loss', color='#ef4444', linewidth=2)
axes[0, 0].plot(vae_history.history['val_loss'], label='Val Loss', color='#f59e0b', linewidth=2, linestyle='--')
axes[0, 0].set_title('VAE Training Loss')
axes[0, 0].set_xlabel('Epoch')
axes[0, 0].set_ylabel('Loss')
axes[0, 0].legend()
axes[0, 0].grid(True, alpha=0.3)

# Reconstruction Error Distribution
axes[0, 1].hist(normal_errors, bins=80, alpha=0.7, label='Normal', color='#0891b2', density=True)
axes[0, 1].hist(anomaly_errors, bins=80, alpha=0.7, label='Anomaly', color='#ef4444', density=True)
axes[0, 1].axvline(x=threshold, color='#f59e0b', linestyle='--', linewidth=2, label=f'Threshold = {threshold:.4f}')
axes[0, 1].set_title('Reconstruction Error Distribution')
axes[0, 1].set_xlabel('Reconstruction Error (MSE)')
axes[0, 1].set_ylabel('Density')
axes[0, 1].legend()
axes[0, 1].set_xlim(0, min(np.percentile(anomaly_errors, 99) * 1.5, 10))

# Confusion Matrix
cm = np.array([[TN, FP], [FN, TP]])
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=axes[1, 0],
            xticklabels=['Normal', 'Anomaly'], yticklabels=['Normal', 'Anomaly'],
            annot_kws={"size": 14})
axes[1, 0].set_title('Confusion Matrix')
axes[1, 0].set_xlabel('Predicted')
axes[1, 0].set_ylabel('Actual')

# Scatter plot of errors
all_errors = np.concatenate([normal_errors[:500], anomaly_errors[:200]])
all_labels = np.concatenate([np.zeros(min(500, len(normal_errors))), np.ones(min(200, len(anomaly_errors)))])
colors_scatter = ['#0891b2' if l == 0 else '#ef4444' for l in all_labels]
axes[1, 1].scatter(range(len(all_errors)), all_errors, c=colors_scatter, alpha=0.5, s=10)
axes[1, 1].axhline(y=threshold, color='#f59e0b', linestyle='--', linewidth=2, label=f'Threshold = {threshold:.4f}')
axes[1, 1].set_title('Reconstruction Errors (Normal=Blue, Anomaly=Red)')
axes[1, 1].set_xlabel('Sample Index')
axes[1, 1].set_ylabel('Reconstruction Error')
axes[1, 1].legend()

plt.suptitle('VAE Anomaly Detection Model Performance', fontsize=16, y=1.02)
plt.tight_layout()
plt.savefig('figures/vae_results.png', dpi=150, bbox_inches='tight')
plt.show()

## 6. Results Summary and Comparison

This section consolidates all experimental results for thesis reporting.

In [ ]:
# ============================================================
# 6.1 Comprehensive Results Summary
# ============================================================

print("=" * 70)
print("  COMPREHENSIVE RESULTS SUMMARY")
print("  Smart RF Spectrum Forecasting & Anomaly Detection")
print("  Zimbabwe's Telecom Networks")
print("=" * 70)

print("\n┌─────────────────────────────────────────────────────┐")
print("│  DATASET STATISTICS                                  │")
print("├─────────────────────────────────────────────────────┤")
print(f"│  Total Records:     {len(df):>10,}                     │")
print(f"│  Cell Towers:       {len(cell_towers):>10}                     │")
print(f"│  Frequency Bands:   {len(frequency_bands):>10}                     │")
print(f"│  Time Range:        Jan 2021 – Dec 2023             │")
print(f"│  Sampling Rate:     1-hour intervals                 │")
print(f"│  Anomaly Rate:      {df['is_anomaly'].mean()*100:>8.2f}%                    │")
print("└─────────────────────────────────────────────────────┘")

print("\n┌─────────────────────────────────────────────────────┐")
print("│  CNN-LSTM FORECASTING MODEL                          │")
print("├─────────────────────────────────────────────────────┤")
print(f"│  Architecture:      Conv1D → LSTM → Dense            │")
print(f"│  Lookback:          {LOOKBACK} hours (3 days)                │")
print(f"│  Horizon:           {FORECAST_HORIZON} hours ahead                    │")
print(f"│  MAE:               {mae_real:>8.4f} GB                     │")
print(f"│  RMSE:              {rmse:>8.4f} GB                     │")
print(f"│  MAPE:              {mape:>8.2f}%                      │")
print("└─────────────────────────────────────────────────────┘")

print("\n┌─────────────────────────────────────────────────────┐")
print("│  VAE ANOMALY DETECTION MODEL                         │")
print("├─────────────────────────────────────────────────────┤")
print(f"│  Architecture:      Encoder → Latent(4) → Decoder    │")
print(f"│  Threshold (P99):   {threshold:>8.4f}                       │")
print(f"│  Precision:         {precision*100:>8.1f}%                      │")
print(f"│  Recall:            {recall*100:>8.1f}%                      │")
print(f"│  F1-Score:          {f1*100:>8.1f}%                      │")
print(f"│  Accuracy:          {accuracy*100:>8.1f}%                      │")
print(f"│  TP: {TP:>5,}  FP: {FP:>5,}  TN: {TN:>5,}  FN: {FN:>5,}      │")
print("└─────────────────────────────────────────────────────┘")

In [ ]:
# ============================================================
# 6.2 Publication-Ready Comparison Table
# ============================================================

# Create a summary DataFrame for thesis Table
summary_data = {
    "Model": ["CNN-LSTM Hybrid", "CNN-LSTM Hybrid", "CNN-LSTM Hybrid", 
              "VAE", "VAE", "VAE", "VAE"],
    "Task": ["Forecasting", "Forecasting", "Forecasting",
             "Anomaly Detection", "Anomaly Detection", "Anomaly Detection", "Anomaly Detection"],
    "Metric": ["MAE (GB)", "RMSE (GB)", "MAPE (%)", 
               "Precision (%)", "Recall (%)", "F1-Score (%)", "Accuracy (%)"],
    "Value": [f"{mae_real:.4f}", f"{rmse:.4f}", f"{mape:.2f}",
              f"{precision*100:.1f}", f"{recall*100:.1f}", f"{f1*100:.1f}", f"{accuracy*100:.1f}"]
}

summary_df = pd.DataFrame(summary_data)
print("\nTable 4.1: Model Performance Summary")
print("=" * 65)
print(summary_df.to_string(index=False))
print("=" * 65)

In [ ]:
# ============================================================
# 6.3 Final Combined Visualization for Thesis
# ============================================================

fig = plt.figure(figsize=(18, 14))

# Row 1: Training curves
ax1 = fig.add_subplot(3, 2, 1)
ax1.plot(history.history['loss'], label='Train', color='#0891b2', linewidth=2)
ax1.plot(history.history['val_loss'], label='Validation', color='#f59e0b', linewidth=2, linestyle='--')
ax1.set_title('(a) CNN-LSTM Training Loss', fontweight='bold')
ax1.set_xlabel('Epoch')
ax1.set_ylabel('MSE Loss')
ax1.legend()
ax1.grid(True, alpha=0.3)

ax2 = fig.add_subplot(3, 2, 2)
ax2.plot(vae_history.history['loss'], label='Train', color='#ef4444', linewidth=2)
ax2.plot(vae_history.history['val_loss'], label='Validation', color='#f59e0b', linewidth=2, linestyle='--')
ax2.set_title('(b) VAE Training Loss', fontweight='bold')
ax2.set_xlabel('Epoch')
ax2.set_ylabel('Loss')
ax2.legend()
ax2.grid(True, alpha=0.3)

# Row 2: Results
ax3 = fig.add_subplot(3, 2, 3)
n_show = 100
ax3.plot(y_test_real[:n_show, 0], label='Actual', color='#0891b2', linewidth=1.5)
ax3.plot(y_pred_real[:n_show, 0], label='Predicted', color='#ef4444', linewidth=1.5, linestyle='--')
ax3.set_title('(c) Forecast: Actual vs Predicted', fontweight='bold')
ax3.set_xlabel('Time Step')
ax3.set_ylabel('Traffic (GB)')
ax3.legend()
ax3.grid(True, alpha=0.3)

ax4 = fig.add_subplot(3, 2, 4)
ax4.hist(normal_errors, bins=60, alpha=0.7, label='Normal', color='#0891b2', density=True)
ax4.hist(anomaly_errors, bins=60, alpha=0.7, label='Anomaly', color='#ef4444', density=True)
ax4.axvline(x=threshold, color='#f59e0b', linestyle='--', linewidth=2, label=f'Threshold')
ax4.set_title('(d) VAE Reconstruction Error', fontweight='bold')
ax4.set_xlabel('Error')
ax4.set_ylabel('Density')
ax4.legend()
ax4.set_xlim(0, min(np.percentile(anomaly_errors, 99) * 1.5, 10))

# Row 3: Metrics
ax5 = fig.add_subplot(3, 2, 5)
forecast_metrics = {'MAE': mae_real, 'RMSE': rmse, 'MAPE': mape/10}
bars = ax5.bar(forecast_metrics.keys(), forecast_metrics.values(), 
               color=['#0891b2', '#10b981', '#f59e0b'], edgecolor='white', width=0.5)
for bar, val, label in zip(bars, [mae_real, rmse, mape], ['GB', 'GB', '%']):
    ax5.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.1, 
             f'{val:.2f} {label}', ha='center', fontweight='bold', fontsize=11)
ax5.set_title('(e) Forecasting Metrics', fontweight='bold')
ax5.set_ylabel('Value')

ax6 = fig.add_subplot(3, 2, 6)
cm = np.array([[TN, FP], [FN, TP]])
sns.heatmap(cm, annot=True, fmt='d', cmap='YlOrRd', ax=ax6,
            xticklabels=['Predicted\nNormal', 'Predicted\nAnomaly'],
            yticklabels=['Actual\nNormal', 'Actual\nAnomaly'],
            annot_kws={"size": 13, "fontweight": "bold"})
ax6.set_title('(f) VAE Confusion Matrix', fontweight='bold')

plt.suptitle('Figure 4.1: Experimental Results — Smart RF Spectrum Forecasting & Anomaly Detection',
             fontsize=15, fontweight='bold', y=1.01)
plt.tight_layout()
plt.savefig('figures/thesis_combined_results.png', dpi=200, bbox_inches='tight')
plt.show()

print("\nAll figures saved to 'figures/' directory.")
print("Notebook execution complete.")

## 7. Conclusion
